In [1]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
CHUNK_CSV = "data_files/chunked_paragraphs.csv"
GITHUB_URL = "https://raw.githubusercontent.com/US-CBO/eval-projections/main/output_data/outlay_projection_errors.csv"

OUTPUT_PARQUET = "data_files/flattened_cbo_reports_synonym_match.parquet"
OUTPUT_CSV = "data_files/chunked_paragraphs_with_embeddings.csv"
SYNONYM_JSON = "data_files/subcategory_synonyms_word2vec.json"

# Generate N nearest-neighbor "synonyms" per label using Word2Vec similar_by_vector
N_SYNONYMS_PER_LABEL = 20

# If True: regenerate synonyms from Word2Vec and OVERWRITE the JSON file
REGENERATE_SYNONYMS = True

# If True: even when loading an existing JSON, rewrite it after applying hygiene cleanup
ALWAYS_REWRITE_JSON_AFTER_CLEANING = True

# ------------------------------------------------------------
# Word2Vec synonym query context
# ------------------------------------------------------------
# These terms are used ONLY to influence the vector passed to similar_by_vector().
# They are NOT automatically added as match phrases.
#
# Conceptually, this makes the synonym query closer to:
#   Medicare spending outlays
#   Social Security spending outlays
#   Other Mandatory spending outlays
#
# But technically, because Word2Vec needs vectors, we combine:
#   vector(subcategory) + vector(spending/outlays)
SYNONYM_CONTEXT_TERMS = ["spending", "outlays","federal","congress","budget"]

# Influence weights:
# Higher label weight = stay closer to Medicare/Social Security/etc.
# Higher context weight = move more toward spending/outlay vocabulary.
#
# 3.0 and 1.0 means:
#   mostly the official subcategory, slightly nudged toward spending/outlays.
SYNONYM_LABEL_WEIGHT = 5.0
SYNONYM_CONTEXT_WEIGHT = 1.0

# Generic junk / dangerous tokens that create false matches
# Keep "spending" and "outlays" here so standalone generic words do not become match terms.
FORBIDDEN_EXACT = {
    "security", "benefits", "insurance", "health",
    "spending", "outlays", "program", "programs",
    "mandatory", "discretionary", "total",
    "net", "interest", "other", "some", "certain", "various", "most", "several", "multiple"
}

FORBIDDEN_SUBSTRINGS = {
    "list", "alternative", "newline", "return", "exactly", "abbreviations"
}

# Extra “wrong security” guardrails: drop neighbors clearly about homeland/transport security
SECURITY_CONTEXT_FORBIDDEN = {
    "homeland", "aviation", "transportation", "tsa", "terror", "terrorism", "border"
}

# ============================================================
# Load paragraphs
# ============================================================
df2 = pd.read_csv(CHUNK_CSV).reset_index(drop=True)
df2["row_id"] = np.arange(len(df2), dtype=np.int64)

for col in ["component", "category", "subcategory", "match_method"]:
    if col not in df2.columns:
        df2[col] = pd.NA

# ============================================================
# Load official CBO mapping + official subcategory list
# ============================================================
df_errors = pd.read_csv(GITHUB_URL)

mapping = (
    df_errors[["subcategory", "category", "component"]]
    .dropna(subset=["subcategory", "category", "component"])
    .drop_duplicates()
    .drop_duplicates(subset=["subcategory"], keep="first")
)

subcategories = sorted(mapping["subcategory"].unique().tolist())
print(f"Loaded {len(subcategories)} official subcategories:\n{subcategories}")

# ============================================================
# Text normalization
#   (1) makes Medi_Cal == Medi-Cal == Medi Cal
#   (2) makes matching ignore '_' and '-' differences
# ============================================================
def normalize_text(x):
    x = "" if pd.isna(x) else str(x).lower()

    # Treat underscores and hyphens as whitespace separators
    x = re.sub(r"[_\-]+", " ", x)

    # Soften other punctuation to spaces while keeping letters/numbers/underscore-class chars
    x = re.sub(r"[^\w\s]", " ", x)

    # Collapse whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x

# ============================================================
# Hygiene filter for formatting/artifact tokens like:
#   -----_Net, =====_Net, ----------------_Total, FINANCIAL_DETAILS, etc.
# ============================================================
def is_formatting_artifact_token(w: str) -> bool:
    if not w:
        return True

    w = str(w)

    # lots of dashes/equals anywhere
    if re.search(r"[-=]{6,}", w):
        return True

    # starts with separator runs
    if re.match(r"^[-=_]{4,}", w):
        return True

    # tokens that are mostly punctuation/underscore
    letters = len(re.findall(r"[A-Za-z]", w))
    digits = len(re.findall(r"\d", w))
    punct = len(w) - letters - digits

    if letters == 0 and digits == 0:
        return True

    if punct / max(len(w), 1) > 0.55:
        return True

    # repeated underscores, common in scraped formatting
    if re.search(r"_{3,}", w):
        return True

    return False

# ============================================================
# Word2Vec synonym generation helpers
# ============================================================
syn_path = Path(SYNONYM_JSON)

def keep_candidate_token(label: str, w: str) -> bool:
    wl = str(w).lower()
    label_l = str(label).lower()

    # Reject formatting/artifact tokens
    if is_formatting_artifact_token(w):
        return False

    # Reject prompt-echo / instruction artifacts
    if any(bad in wl for bad in FORBIDDEN_SUBSTRINGS):
        return False

    # Reject exact generic junk
    if wl in FORBIDDEN_EXACT:
        return False

    # Reject numeric / weird tokens
    if re.fullmatch(r"\d+", str(w)):
        return False

    if len(str(w)) <= 2:
        return False

    if re.search(r"\d", str(w)):
        return False

    # Prevent overlap across program labels.
    # Example: avoid "Medicaid" showing up as a Medicare synonym.
    for other in subcategories:
        if other.lower() == label_l:
            continue

        other_norm = normalize_text(other).replace(" ", "_")

        if other_norm and other_norm in wl:
            return False

    # Social Security: eliminate homeland/transport security drift and "security" alone
    if label == "Social Security":
        if wl == "security":
            return False

        if any(tok in wl for tok in SECURITY_CONTEXT_FORBIDDEN):
            return False

    return True


def label_to_vector(label: str, model):
    """
    Build a vector for the official label only.

    1) Try underscore phrase token:
           Social Security -> Social_Security

    2) Else average vectors for words in the label:
           Social + Security

    Note:
    This function does NOT add spending/outlays context.
    That is handled separately by add_context_influence_to_vector().
    """
    label = str(label).strip()

    phrase_token = label.replace(" ", "_")

    if phrase_token in model:
        return model[phrase_token].astype(np.float32)

    parts = label.split()

    vecs = [model[p].astype(np.float32) for p in parts if p in model]

    if not vecs:
        vecs = [model[p.title()].astype(np.float32) for p in parts if p.title() in model]

    if not vecs:
        return None

    return np.mean(np.vstack(vecs), axis=0).astype(np.float32)


def unit_vec(v):
    """
    Normalize vector length so influence weights behave more predictably.
    """
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)

    if n == 0:
        return v

    return v / n


def add_context_influence_to_vector(v, model):
    """
    Convert a label vector into a contextual synonym-generation vector.

    Conceptually approximates:

        [subcategory] spending outlays

    with influence weights.

    Example:
        If SYNONYM_LABEL_WEIGHT = 3.0
        and SYNONYM_CONTEXT_WEIGHT = 1.0,

        final_vector = 3 * subcategory_vector
                     + 1 * average(spending_vector, outlays_vector)

    This affects ONLY Word2Vec nearest-neighbor synonym generation.
    It does NOT add 'spending outlays' phrases to your matcher.
    """
    context_vecs = []

    for term in SYNONYM_CONTEXT_TERMS:
        if term in model:
            context_vecs.append(model[term].astype(np.float32))

    # If spending/outlays are unavailable, return original vector
    if not context_vecs:
        return unit_vec(v).astype(np.float32)

    context_vec = np.mean(np.vstack(context_vecs), axis=0).astype(np.float32)

    combined = (
        SYNONYM_LABEL_WEIGHT * unit_vec(v)
        + SYNONYM_CONTEXT_WEIGHT * unit_vec(context_vec)
    )

    return unit_vec(combined).astype(np.float32)

# ============================================================
# Generate or load synonyms
# ============================================================
if (not syn_path.exists()) or REGENERATE_SYNONYMS:
    print(
        "\nGenerating synonyms using Word2Vec similar_by_vector "
        f"with context terms {SYNONYM_CONTEXT_TERMS} "
        f"and weights label={SYNONYM_LABEL_WEIGHT}, context={SYNONYM_CONTEXT_WEIGHT} "
        "(OVERWRITING JSON)..."
    )

    import gensim.downloader as api

    w2v_model = api.load("word2vec-google-news-300")

    synonyms = {}
    debug_kept = {}

    for sub in subcategories:
        # Base vector for official label:
        #   Medicare
        #   Social_Security, if available
        #   Other + Mandatory, if phrase token unavailable
        v = label_to_vector(sub, w2v_model)

        if v is None:
            synonyms[sub] = [sub]
            debug_kept[sub] = 0
            continue

        # Add spending/outlays influence ONLY for synonym generation.
        # This is the vector equivalent of querying:
        #   "[subcategory] spending outlays"
        v = add_context_influence_to_vector(v, w2v_model)

        nbrs = w2v_model.similar_by_vector(v, topn=1200)

        kept = []

        for w, score in nbrs:
            # Drop the label token itself
            if w.lower() in {sub.lower(), sub.replace(" ", "_").lower()}:
                continue

            if not keep_candidate_token(sub, w):
                continue

            kept.append(w)

            if len(kept) >= N_SYNONYMS_PER_LABEL:
                break

        synonyms[sub] = [sub] + kept
        debug_kept[sub] = len(kept)

    syn_path.parent.mkdir(parents=True, exist_ok=True)
    syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")

    print(f"Saved synonyms to: {SYNONYM_JSON}")

    print("\nNeighbors kept per subcategory:")
    print(pd.Series(debug_kept).sort_values(ascending=False))

else:
    print(f"\nLoading synonyms from: {SYNONYM_JSON}")

    synonyms = json.loads(syn_path.read_text(encoding="utf-8"))

    # Apply hygiene cleanup even when loading, and optionally rewrite the JSON file
    cleaned = {}

    for sub, syns in synonyms.items():
        kept = []

        for w in syns:
            # Keep official label always
            if str(w).strip().lower() == str(sub).lower():
                kept.append(sub)
                continue

            if keep_candidate_token(sub, str(w)):
                kept.append(str(w))

        # De-dupe preserve order
        seen = set()
        out = []

        for w in kept:
            k = str(w).lower()

            if k in seen:
                continue

            seen.add(k)
            out.append(w)

        cleaned[sub] = out

    synonyms = cleaned

    if ALWAYS_REWRITE_JSON_AFTER_CLEANING:
        syn_path.parent.mkdir(parents=True, exist_ok=True)
        syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")
        print(f"Rewrote cleaned synonyms to: {SYNONYM_JSON}")

# ============================================================
# Build phrase match patterns: official labels + Word2Vec neighbors
#   - uses normalize_text() so Medi_Cal matches Medi-Cal, etc.
#   - note: spending/outlays context was used only for synonym generation
# ============================================================
phrase_records = []

for sub, syns in synonyms.items():
    for phrase in syns:
        pnorm = normalize_text(phrase)

        if not pnorm:
            continue

        if pnorm in FORBIDDEN_EXACT:
            continue

        # Also drop phrases that normalize into formatting artifacts
        if is_formatting_artifact_token(phrase):
            continue

        phrase_records.append((phrase, sub))

# Longest-first so more specific phrases win
phrase_records = sorted(phrase_records, key=lambda x: len(str(x[0])), reverse=True)

compiled = []

for phrase, sub in phrase_records:
    pnorm = normalize_text(phrase)

    pat = re.compile(
        r"(?<!\w)" + re.escape(pnorm) + r"(?!\w)",
        flags=re.IGNORECASE
    )

    compiled.append((pat, sub, phrase))


def match_by_phrases(text: str):
    t = normalize_text(text)

    for pat, sub, phrase in compiled:
        if pat.search(t):
            return sub, phrase

    return pd.NA, pd.NA

# ============================================================
# Apply matching: direct phrase hits only
# ============================================================
print("\nMatching paragraphs by direct phrase hits: official labels + Word2Vec neighbors...")

matched_subs = []
matched_phrase = []

for txt in df2["text"].astype(str):
    sub, phrase = match_by_phrases(txt)

    matched_subs.append(sub)
    matched_phrase.append(phrase)

df2["subcategory"] = matched_subs
df2["matched_phrase"] = matched_phrase

df2["match_method"] = np.where(
    df2["subcategory"].notna(),
    "direct_phrase_w2v_neighbors",
    pd.NA
)

# Attach category/component
df2 = (
    df2
    .drop(columns=["category", "component"], errors="ignore")
    .merge(mapping, on="subcategory", how="left")
)

print("\nMatch counts:")
print(df2["match_method"].value_counts(dropna=False))

print("\nTop subcategories:")
print(df2["subcategory"].value_counts(dropna=False).head(15))

# ============================================================
# Save outputs
# ============================================================
Path(OUTPUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)

print(f"\nWriting parquet: {OUTPUT_PARQUET}")
df2.to_parquet(OUTPUT_PARQUET, index=False, engine="pyarrow", compression="zstd")

print(f"Writing CSV: {OUTPUT_CSV}")
df2.to_csv(OUTPUT_CSV, index=False)

print("Done.")

Loaded 10 official subcategories:
['Defense Discretionary', 'Medicaid', 'Medicare', 'Net Interest', 'Nondefense Discretionary', 'Other Mandatory', 'Social Security', 'Total', 'Total Discretionary', 'Total Mandatory']

Generating synonyms using Word2Vec similar_by_vector with context terms ['spending', 'outlays', 'federal', 'congress', 'budget'] and weights label=5.0, context=1.0 (OVERWRITING JSON)...
Saved synonyms to: data_files/subcategory_synonyms_word2vec.json

Neighbors kept per subcategory:
Defense Discretionary       20
Medicaid                    20
Medicare                    20
Net Interest                20
Nondefense Discretionary    20
Other Mandatory             20
Social Security             20
Total                       20
Total Discretionary         20
Total Mandatory             20
dtype: int64

Matching paragraphs by direct phrase hits: official labels + Word2Vec neighbors...

Match counts:
match_method
direct_phrase_w2v_neighbors    20321
<NA>                      